# Barebones SpatialData Image Write Test (`tiffslide` zarr store, merged OME-TIFF)

This notebook mirrors the working `generate_spatialdata_v3-SLIDE-329.ipynb` pattern as closely as possible, but for a single merged OME-TIFF.

It does only five things:

1. inspect the merged OME-TIFF metadata and TIFF tiling
2. open the merged image with `tiffslide` and inspect the zarr-backed view
3. build a multiscale `DataTree` directly from the existing TIFF pyramid levels
4. wrap it in `SpatialData(images={...})` and write it
5. reopen the written store and inspect the result

The key question is whether this avoids the bad chunking/memory behavior seen with plane-based Dask readers.


In [15]:
from __future__ import annotations

import os
import json
import shutil
from pathlib import Path

os.environ.setdefault("NUMBA_CACHE_DIR", "/tmp/numba_cache")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/mplconfig")

import dask.array as da
import spatialdata
import tifffile
import tiffslide
import xarray as xr
from spatialdata import SpatialData, read_zarr
from spatialdata.transformations import Identity, Scale
from xarray import DataArray, Dataset, DataTree

print("spatialdata", spatialdata.__version__)
print("tifffile", tifffile.__version__)
print("tiffslide", tiffslide.__version__)
print("xarray", xr.__version__)


spatialdata 0.7.2
tifffile 2026.3.3
tiffslide 3.0.0
xarray 2026.4.0


In [16]:
OUTPUTS_DIR = Path("/mnt/c/Analysis/Data_prototype/SLIDE-0329/outputs")
FULL_MERGE_PATH = Path("/mnt/c/Analysis/Data_prototype/SLIDE-0329_full_merge.ome.tif")
CHANNEL_MAP_PATH = Path("/mnt/c/Analysis/Data_prototype/SLIDE-0329/outputs/channel_map.generated.json")
WRITE_PATH = OUTPUTS_DIR / "spatialdata_image_write_only_tiffslide_zarr_merged_SLIDE-0329.sdata.zarr"
CHUNK_SHAPE = (1, 512, 512)
IMAGE_LAYER = "full_image"
COORDINATE_SYSTEM = "global"

print(FULL_MERGE_PATH)
print(CHANNEL_MAP_PATH)
print(WRITE_PATH)
print({
    "chunk_shape": CHUNK_SHAPE,
    "image_layer": IMAGE_LAYER,
    "coordinate_system": COORDINATE_SYSTEM,
})


/mnt/c/Analysis/Data_prototype/SLIDE-0329_full_merge.ome.tif
/mnt/c/Analysis/Data_prototype/SLIDE-0329/outputs/channel_map.generated.json
/mnt/c/Analysis/Data_prototype/SLIDE-0329/outputs/spatialdata_image_write_only_tiffslide_zarr_merged_SLIDE-0329.sdata.zarr
{'chunk_shape': (1, 512, 512), 'image_layer': 'full_image', 'coordinate_system': 'global'}


In [17]:
channel_map = json.loads(CHANNEL_MAP_PATH.read_text())
channel_aliases = [entry["alias"] for entry in channel_map]

with tifffile.TiffFile(FULL_MERGE_PATH) as tif:
    series = tif.series[0]
    level0 = series.levels[0]
    page0 = level0.pages[0]
    tile_info = {
        "is_tiled": page0.is_tiled,
        "tilewidth": page0.tilewidth,
        "tilelength": page0.tilelength,
        "compression": str(page0.compression),
    }
    level_shapes = [level.shape[-2:] for level in series.levels]
    print("series axes:", series.axes)
    print("series shape:", series.shape)
    print("levels:", len(series.levels))
    print("level0 shape:", level0.shape)
    print("level_shapes:", level_shapes)
    print("tile_info:", tile_info)

print("channel count in channel map:", len(channel_aliases))
print("first aliases:", channel_aliases[:6])


series axes: CYX
series shape: (25, 62617, 66406)
levels: 8
level0 shape: (25, 62617, 66406)
level_shapes: [(62617, 66406), (31308, 33203), (15654, 16601), (7827, 8300), (3913, 4150), (1956, 2075), (978, 1037), (489, 518)]
tile_info: {'is_tiled': True, 'tilewidth': 256, 'tilelength': 256, 'compression': '8'}
channel count in channel map: 25
first aliases: ['R1_DAPI_AF', 'R1_BIT2_RS0584_CY3B', 'R1_BIT3_RS0015_CY5', 'R1_BIT4_RS0083_750', 'R1_DAPI', 'R1_BIT1_RS0996_488']


In [18]:
slide = tiffslide.open_slide(str(FULL_MERGE_PATH))
zarr_store = slide.zarr_group.store
zarr_img = xr.open_zarr(zarr_store, consolidated=False, mask_and_scale=False)

metadata = {
    "dimensions": slide.dimensions,
    "level_count": slide.level_count,
    "level_dimensions": slide.level_dimensions,
    "level_downsamples": slide.level_downsamples,
}

print("slide metadata:", metadata)
print("zarr keys:", list(zarr_img.keys()))
for key in sorted(zarr_img.keys(), key=int):
    arr = zarr_img[key]
    print(key, arr.dims, arr.shape, getattr(arr.data, "chunks", None))


slide metadata: {'dimensions': (66406, 62617), 'level_count': 8, 'level_dimensions': ((66406, 62617), (33203, 31308), (16601, 15654), (8300, 7827), (4150, 3913), (2075, 1956), (1037, 978), (518, 489)), 'level_downsamples': (1.0, 2.000015970359014, 4.000092178053128, 8.000425327219187, 16.001872904344186, 32.00783637617957, 64.0311032690256, 128.12401797064373)}
zarr keys: ['2', '0', '7', '6', '1', '4', '5', '3']
0 ('C', 'Y', 'X') (25, 62617, 66406) ((1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1), (256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256, 256

In [21]:
zarr_img

<xarray.Dataset> Size: 277GB
Dimensions:  (C: 25, Y2: 15654, X2: 16601, Y: 62617, X: 66406, Y7: 489,
              X7: 518, Y6: 978, X6: 1037, Y1: 31308, X1: 33203, Y4: 3913,
              X4: 4150, Y5: 1956, X5: 2075, Y3: 7827, X3: 8300)
Dimensions without coordinates: C, Y2, X2, Y, X, Y7, X7, Y6, X6, Y1, X1, Y4,
                                X4, Y5, X5, Y3, X3
Data variables:
    2        (C, Y2, X2) uint16 13GB dask.array<chunksize=(1, 256, 256), meta=np.ndarray>
    0        (C, Y, X) uint16 208GB dask.array<chunksize=(1, 256, 256), meta=np.ndarray>
    7        (C, Y7, X7) uint16 13MB dask.array<chunksize=(1, 256, 256), meta=np.ndarray>
    6        (C, Y6, X6) uint16 51MB dask.array<chunksize=(1, 256, 256), meta=np.ndarray>
    1        (C, Y1, X1) uint16 52GB dask.array<chunksize=(1, 256, 256), meta=np.ndarray>
    4        (C, Y4, X4) uint16 812MB dask.array<chunksize=(1, 256, 256), meta=np.ndarray>
    5        (C, Y5, X5) uint16 203MB dask.array<chunksize=(1, 256, 256), meta=np.ndarray>
    3        (C, Y3, X3) uint16 3GB dask.array<chunksize=(1, 256, 256), meta=np.ndarray>
Attributes:
    multiscales:  [{'datasets': [{'path': '0'}, {'path': '1'}, {'path': '2'},...

In [26]:
levels = [zarr_img[str(k)] for k in sorted(zarr_img.keys(), key=int)]
base_shape = levels[0].shape[-2:]

images = {}
for level, arr in enumerate(levels):
    print(f"level {level} raw dims={arr.dims} shape={arr.shape}")
    arr = arr.squeeze(drop=True)

    rename_map = {}
    for dim in arr.dims:
        if dim.lower().startswith("c"):
            rename_map[dim] = "c"
        elif dim.lower().startswith("y"):
            rename_map[dim] = "y"
        elif dim.lower().startswith("x"):
            rename_map[dim] = "x"
    if rename_map:
        arr = arr.rename(rename_map)

    if arr.ndim == 2:
        arr = arr.expand_dims(dim={"c": [channel_aliases[0]]})
    elif arr.ndim != 3:
        raise ValueError(f"Unexpected level {level} dims after squeeze/rename: {arr.dims}")

    expected_dims = {"c", "y", "x"}
    if set(arr.dims) != expected_dims:
        raise ValueError(f"Unexpected level {level} dims after normalization: {arr.dims}")

    arr = arr.transpose("c", "y", "x")
    print(f"level {level} normalized dims={arr.dims} shape={arr.shape}")

    arr = arr.chunk({"c": CHUNK_SHAPE[0], "y": CHUNK_SHAPE[1], "x": CHUNK_SHAPE[2]})
    arr.coords["c"] = channel_aliases[: arr.shape[0]]

    scale_y = base_shape[0] / arr.shape[-2]
    scale_x = base_shape[1] / arr.shape[-1]
    transform = (
        Scale([scale_y, scale_x], axes=("y", "x"))
        if (scale_y != 1.0 or scale_x != 1.0)
        else Identity()
    )
    arr.attrs["transform"] = {COORDINATE_SYSTEM: transform}

    images[f"scale{level}"] = Dataset({"image": arr})

tree = DataTree.from_dict(images)
sdata = SpatialData(images={IMAGE_LAYER: tree})
sdata


level 0 raw dims=('C', 'Y', 'X') shape=(25, 62617, 66406)
level 0 normalized dims=('c', 'y', 'x') shape=(25, 62617, 66406)
level 1 raw dims=('C', 'Y1', 'X1') shape=(25, 31308, 33203)
level 1 normalized dims=('c', 'y', 'x') shape=(25, 31308, 33203)
level 2 raw dims=('C', 'Y2', 'X2') shape=(25, 15654, 16601)
level 2 normalized dims=('c', 'y', 'x') shape=(25, 15654, 16601)
level 3 raw dims=('C', 'Y3', 'X3') shape=(25, 7827, 8300)
level 3 normalized dims=('c', 'y', 'x') shape=(25, 7827, 8300)
level 4 raw dims=('C', 'Y4', 'X4') shape=(25, 3913, 4150)
level 4 normalized dims=('c', 'y', 'x') shape=(25, 3913, 4150)
level 5 raw dims=('C', 'Y5', 'X5') shape=(25, 1956, 2075)
level 5 normalized dims=('c', 'y', 'x') shape=(25, 1956, 2075)
level 6 raw dims=('C', 'Y6', 'X6') shape=(25, 978, 1037)
level 6 normalized dims=('c', 'y', 'x') shape=(25, 978, 1037)
level 7 raw dims=('C', 'Y7', 'X7') shape=(25, 489, 518)
level 7 normalized dims=('c', 'y', 'x') shape=(25, 489, 518)


SpatialData object
└── Images
      └── 'full_image': DataTree[cyx] (25, 62617, 66406), (25, 31308, 33203), (25, 15654, 16601), (25, 7827, 8300), (25, 3913, 4150), (25, 1956, 2075), (25, 978, 1037), (25, 489, 518)
with coordinate systems:
    ▸ 'global', with elements:
        full_image (Images)

In [27]:
scale0 = sdata.images[IMAGE_LAYER]["scale0"].image
print("scale0 shape:", scale0.shape)
print("scale0 dims:", scale0.dims)
print("scale0 chunks:", scale0.chunks)
print("scale0 data chunksize:", getattr(scale0.data, "chunksize", None))
print("scale0 channel coords:", scale0.coords["c"].values[:6].tolist())


scale0 shape: (25, 62617, 66406)
scale0 dims: ('c', 'y', 'x')
scale0 chunks: ((1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1), (512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 153), (512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 51

In [28]:
sdata

SpatialData object
└── Images
      └── 'full_image': DataTree[cyx] (25, 62617, 66406), (25, 31308, 33203), (25, 15654, 16601), (25, 7827, 8300), (25, 3913, 4150), (25, 1956, 2075), (25, 978, 1037), (25, 489, 518)
with coordinate systems:
    ▸ 'global', with elements:
        full_image (Images)

In [29]:
if WRITE_PATH.exists():
    shutil.rmtree(WRITE_PATH)

sdata.write(WRITE_PATH, overwrite=True)
print("Wrote:", WRITE_PATH)
print("Exists:", WRITE_PATH.exists())


/home/ratnayn/miniconda3/envs/sdata_tiffslide/lib/python3.11/site-packages/ome_zarr/writer.py:319: FutureWarning: Passing storage-related arguments via **kwargs is deprecated. Please use the 'zarr_store_kwargs' parameter instead. **kwargs will be removed in a future version.
  da_delayed = da.to_zarr(


Wrote: /mnt/c/Analysis/Data_prototype/SLIDE-0329/outputs/spatialdata_image_write_only_tiffslide_zarr_merged_SLIDE-0329.sdata.zarr
Exists: True


In [30]:
reloaded = read_zarr(WRITE_PATH)
print(reloaded)
print("images", list(reloaded.images.keys()))
print("labels", list(reloaded.labels.keys()))
print("tables", list(reloaded.tables.keys()))


SpatialData object, with associated Zarr store: /mnt/c/Analysis/Data_prototype/SLIDE-0329/outputs/spatialdata_image_write_only_tiffslide_zarr_merged_SLIDE-0329.sdata.zarr
└── Images
      └── 'full_image': DataTree[cyx] (25, 62617, 66406), (25, 31308, 33203), (25, 15654, 16601), (25, 7827, 8300), (25, 3913, 4150), (25, 1956, 2075), (25, 978, 1037), (25, 489, 518)
with coordinate systems:
    ▸ 'global', with elements:
        full_image (Images)
images ['full_image']
labels []
tables []
